# ML-02 — Research Question and Provisional Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/harshit5445/flyrank-ml-internship/blob/main/work/notebooks/w01_research_question.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane (or freestyle) and why

**Lane 2 — Refresh / Content Opportunity Scoring.**

I chose this lane because the starter data contains page-level search and content signals such as impressions, clicks, CTR, average position, content type, word count, freshness, and trend direction. The practical question is not simply whether a page is declining; it is which pages should be reviewed first when a team has limited time. This lane lets me turn observable signals into a ranked review queue with reason codes and then test whether a simple ranking can prioritize useful candidates. The starter pipeline already demonstrates this workflow, but my goal is to build and validate my own decision-support approach rather than copy an existing product score.


In [9]:
!git clone https://github.com/harshit5445/flyrank-ml-internship.git
%cd /content/flyrank-ml-internship


import pandas as pd

DATA_PATH = "data/raw/content_refresh_anonymized.csv"
df = pd.read_csv(DATA_PATH)

print(f"Rows: {len(df):,}")
print(f"Columns: {df.shape[1]:,}")
print(f"Available fields include: impressions_90d, ctr, avg_position, trend_direction, content_type")


Cloning into 'flyrank-ml-internship'...
remote: Enumerating objects: 119, done.
remote: Counting objects: 100% (119/119), done.
remote: Compressing objects: 100% (75/75), done.
remote: Total 119 (delta 34), reused 95 (delta 28), pack-reused 0 (from 0)
Receiving objects: 100% (119/119), 1.85 MiB | 12.11 MiB/s, done.
Resolving deltas: 100% (34/34), done.
/content/flyrank-ml-internship
Rows: 30,000
Columns: 44
Available fields include: impressions_90d, ctr, avg_position, trend_direction, content_type


## 2. The question: decision, action, cost of a wrong call

**Decision:** Which pages should a content/search reviewer inspect first for a possible refresh, expansion, protection, pruning, or monitoring action?

**Who acts:** A content or SEO reviewer uses the ranked queue to decide where to spend limited review time.

**Action:** For each high-priority page, the system should provide a rank, a suggested review context, and understandable reason codes such as declining movement, meaningful exposure, position, CTR, or freshness.

**Cost of a wrong call:** A false positive can waste editorial/SEO time on a page that does not need intervention. A false negative can cause a genuinely weakening page to be missed. The project therefore needs to balance precision at the top of the queue with the cost of missing useful candidates. A high score should mean “review this first,” not “this page is guaranteed to recover if changed.”


In [10]:
# A simple check of the decision variables available in the starter data.
required = ["content_id", "impressions_90d", "ctr", "avg_position", "trend_direction"]
missing = [c for c in required if c not in df.columns]

print("Required decision-support fields present:", len(missing) == 0)
if missing:
    print("Missing:", missing)
else:
    print("Decision grain: one content/page record")
    print("Potential outcome signal:", "trend_direction")
    print("Potential exposure signal:", "impressions_90d")


Required decision-support fields present: True
Decision grain: one content/page record
Potential outcome signal: trend_direction
Potential exposure signal: impressions_90d


## 3. Quick look at the data (2-3 real numbers)

The starter dataset contains **30,000 rows and 44 columns**. The starter discovery also shows that position and CTR are strongly different across position tiers: mean CTR is **0.3548 for page 1** and **0.0554 for deep results** among pages with at least 100 impressions. This makes ranking and prioritization worth investigating because the meaning of a weak CTR or weak performance depends on the page's search position.

The starter pipeline also reports a **Precision@50 of 0.240 for its hand-written baseline and 0.740 for its random-forest model** on the 30,000-row starter slice using client-holdout validation. I treat this as evidence that a ranked review queue is a meaningful decision-support problem on this slice, not as proof that the same performance will hold on the full warehouse.


In [11]:
# Reproduce the key starter-data numbers used in this framing.
print(f"Dataset size: {df.shape[0]:,} rows × {df.shape[1]:,} columns")

visible = df[df["impressions_90d"] >= 100]
ctr_by_pos = visible.groupby("position_tier")["ctr"].mean().sort_values(ascending=False)

print(f"Mean CTR, page_1: {ctr_by_pos['page_1']:.4f}")
print(f"Mean CTR, deep:   {ctr_by_pos['deep']:.4f}")

# Starter-pipeline reference numbers, included as context rather than as new claims.
baseline_p50 = 0.240
rf_p50 = 0.740
print(f"Starter baseline Precision@50: {baseline_p50:.3f}")
print(f"Starter random-forest Precision@50: {rf_p50:.3f}")


Dataset size: 30,000 rows × 44 columns
Mean CTR, page_1: 0.3548
Mean CTR, deep:   0.0554
Starter baseline Precision@50: 0.240
Starter random-forest Precision@50: 0.740


## 4. Careful words: what I can and can't claim

I can report **observed** relationships and **directional** patterns in the anonymized data. I can build **decision-support** that ranks pages for human review and evaluate how well the ranking identifies pages matching a clearly defined outcome.

I cannot claim that a correlation proves causation, that changing a page will definitely recover traffic, or that I have discovered or predicted a Google ranking algorithm. I also cannot treat the starter `trend_direction == "down"` label as a perfect future outcome; the lane guide describes it as a beginner proxy based on the current window. For a stronger capstone, I should define a leakage-safe future outcome and validate it with an appropriate client or time-based split.

The starter Precision@50 result is limited to the 30,000-row anonymized starter slice and its client-holdout validation. It should not be presented as a benchmark for the full warehouse.


In [12]:
# Final framing checks
assert len(df) == 30000
assert df.shape[1] == 44
assert "trend_direction" in df.columns
assert "impressions_90d" in df.columns
assert "ctr" in df.columns

print("Framing checks passed.")
print("Claims are limited to observed/directional/decision-support language.")
print("No client names, raw URLs, private queries, or causal claims are used.")


Framing checks passed.
Claims are limited to observed/directional/decision-support language.
No client names, raw URLs, private queries, or causal claims are used.


## Self-check

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.